# Stg Layer — restaurant_geocode (Google Places API, cached)

Look up each restaurant extracted from the videos (`stg_restaurant_mentions`) with **Places API (New) Text Search** and cache the answer in `${silver}.stg_restaurant_geocode`, one row per restaurant key = normalized name + normalized district (the same key `gold/restaurants` uses).

**Flow:**
1. Distinct restaurant keys from `stg_restaurant_mentions` (representative name / address = latest mention with an address)
2. Keys **not yet in the cache** → call the API, within the **daily cap** (`DAILY_CALL_LIMIT` calls per Vietnam day, counted over every run of the day in `stg_restaurant_geocode_calls`); the rest wait for the next day
3. Pick the first candidate that is a food place (`restaurant`, `food`, `cafe`, `bakery`, `*_restaurant`, ...) **and** has a similar name (unaccented token overlap ≥ `MIN_NAME_SIMILARITY`); otherwise store NULLs with a `match_status` (`no_result`, `not_food`, `name_mismatch`)
4. MERGE into the cache (insert-only on the key). HTTP errors are **not** cached, so they are retried next run.

**Quota:** on HTTP 429 (e.g. `SearchTextRequest per day` used up) the run stops calling the API but does **not** fail: what was found is cached, the rest waits for the next run, and Gold uses the name + district fallback meanwhile. A rejected key (401 / 403) fails the task.

**Address fields:** Google returns the post-2025 wards (e.g. "Vĩnh Hội") and no district, so `ward` / `city` / `latitude` / `longitude` / `place_id` / `formatted_address` come from Google and the district stays the one stated in the video.

**Secret:** `maps/api_key` (Places API (New)). Uses `requests` (no psycopg, no FIPS issue).

**Cost:** the field mask only asks for Pro-tier fields (id, name, address, location, types, address components), so each call is billed as **Text Search Pro**: 5,000 free calls per month, then $32 per 1,000. Asking for `rating` / `userRatingCount` would make every call **Enterprise** (1,000 free, then $35 per 1,000). The daily cap of 100 calls (≈ 3,000 per month) keeps the bill at $0 even if the Google project's own quota is higher.


In [ ]:
%run ../gold/_udfs


In [ ]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

MENTIONS_TABLE = f"{SILVER}.stg_restaurant_mentions"
GEOCODE_TABLE = f"{SILVER}.stg_restaurant_geocode"
CALLS_TABLE = f"{SILVER}.stg_restaurant_geocode_calls"  # API calls per run, for the daily cap
DAILY_CALL_LIMIT = 100          # Places API calls per Vietnam day, over all runs (cost guard)
SNAPSHOT_TZ = "Asia/Ho_Chi_Minh"
MIN_NAME_SIMILARITY = 0.5
HCMC_BIAS = {"circle": {"center": {"latitude": 10.7769, "longitude": 106.7009}, "radius": 30000.0}}
# Pro-tier fields only: adding rating / userRatingCount would bill every call as Text Search Enterprise
# (1,000 free calls / month instead of 5,000). rating / rating_count stay in the cache table as NULL.
FIELD_MASK = ("places.id,places.displayName,places.formattedAddress,places.location,places.types,"
              "places.addressComponents")
FOOD_TYPES = {"restaurant", "food", "cafe", "coffee_shop", "bakery", "bar", "meal_takeaway", "meal_delivery",
              "food_court", "dessert_shop", "ice_cream_shop", "tea_house", "juice_shop", "snack_bar", "diner",
              "bistro", "deli", "pub"}

print(f"Mentions source: {MENTIONS_TABLE}")
print(f"Geocode cache: {GEOCODE_TABLE} · max {DAILY_CALL_LIMIT} API calls per day")


In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GEOCODE_TABLE} (
        restaurant_name_normalized STRING,
        district_norm              STRING,
        query_text                 STRING,
        match_status               STRING,
        place_id                   STRING,
        place_name                 STRING,
        formatted_address          STRING,
        ward                       STRING,
        city                       STRING,
        latitude                   DOUBLE,
        longitude                  DOUBLE,
        place_types                ARRAY<STRING>,
        rating                     DOUBLE,
        rating_count               INT,
        name_similarity            DOUBLE,
        geocoded_at                TIMESTAMP
    )
    USING DELTA
    COMMENT 'Google Places lookup per restaurant key (normalized name + district). match_status = matched / no_result / not_food / name_mismatch.'
""")

# --- Restaurant keys not in the cache yet (district_norm = same rule as gold/restaurants) ---
todo = spark.sql(rf"""
    WITH m AS (
        SELECT *,
            nullif(trim(regexp_replace(regexp_replace(regexp_replace(
                lower(unaccent_vi(coalesce(district, ''))),
                '^(q\\.?\\s*|quan\\s+)0*(\\d+)$', 'quan $2'),
                '^(tp\\.?|thanh pho)\\s+', ''),
                '\\s+', ' ')), '') AS district_norm,
            ROW_NUMBER() OVER (
                PARTITION BY restaurant_name_normalized,
                    nullif(trim(regexp_replace(regexp_replace(regexp_replace(
                        lower(unaccent_vi(coalesce(district, ''))),
                        '^(q\\.?\\s*|quan\\s+)0*(\\d+)$', 'quan $2'),
                        '^(tp\\.?|thanh pho)\\s+', ''),
                        '\\s+', ' ')), '')
                ORDER BY address IS NULL, extracted_at DESC, video_id) AS rn
        FROM {MENTIONS_TABLE}
    )
    SELECT m.restaurant_name_normalized, m.district_norm, m.restaurant_name, m.address, m.district, m.city
    FROM m
    LEFT ANTI JOIN {GEOCODE_TABLE} g
      ON g.restaurant_name_normalized = m.restaurant_name_normalized AND g.district_norm <=> m.district_norm
    WHERE m.rn = 1
    ORDER BY m.restaurant_name_normalized
""").collect()

# --- Daily cap: calls already made today (Vietnam day) by earlier runs ---
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CALLS_TABLE} (
        run_ts    TIMESTAMP,
        call_date DATE,
        api_calls INT,
        cached    INT,
        errors    INT
    )
    USING DELTA
    COMMENT 'Places API calls per geocode run; the sum per call_date enforces DAILY_CALL_LIMIT.'
""")
today = spark.sql(f"SELECT to_date(from_utc_timestamp(current_timestamp(), '{SNAPSHOT_TZ}')) AS d").collect()[0]["d"]
used_today = spark.sql(f"SELECT COALESCE(SUM(api_calls), 0) AS n FROM {CALLS_TABLE} WHERE call_date = DATE'{today}'").collect()[0]["n"]
budget = max(0, DAILY_CALL_LIMIT - used_today)
print(f"Restaurant keys not in the cache: {len(todo)} · calls today so far: {used_today}/{DAILY_CALL_LIMIT} · this run may call: {budget}")


In [ ]:
# --- Call Places API (New) Text Search and pick the first acceptable candidate ---
import difflib
import re

import requests

API_KEY = dbutils.secrets.get("maps", "api_key")


def name_tokens(s):
    return set(re.sub(r"[^a-z0-9 ]", " ", (unaccent_vi(s or "")).lower()).split())


def name_similarity(ours, theirs):
    """Max of the character ratio and the share of our tokens found in their name (unaccented)."""
    a = " ".join(sorted(name_tokens(ours)))
    b = " ".join(sorted(name_tokens(theirs)))
    ours_t, theirs_t = name_tokens(ours), name_tokens(theirs)
    overlap = len(ours_t & theirs_t) / len(ours_t) if ours_t else 0.0
    return max(difflib.SequenceMatcher(None, a, b).ratio(), overlap)


def component(place, *types):
    for c in place.get("addressComponents", []):
        if any(t in c.get("types", []) for t in types):
            return c.get("longText")
    return None


def ward_from_address(formatted_address):
    """Post-2025 addresses end with '<ward>, <city>': Google often has the ward only in the text."""
    parts = [p.strip() for p in (formatted_address or "").split(",")]
    return parts[-2] if len(parts) >= 3 else None


def geocode(row):
    city = row["city"] or "Hồ Chí Minh"
    parts = [row["restaurant_name"], row["address"], row["district"], city]
    query = ", ".join(p for p in parts if p)
    body = {"textQuery": query, "languageCode": "vi", "regionCode": "VN", "pageSize": 5}
    if "chi minh" in unaccent_vi(city).lower():
        body["locationBias"] = HCMC_BIAS
    resp = requests.post(
        "https://places.googleapis.com/v1/places:searchText",
        headers={"X-Goog-Api-Key": API_KEY, "X-Goog-FieldMask": FIELD_MASK},
        json=body, timeout=30,
    )
    if resp.status_code != 200:
        return None, f"HTTP {resp.status_code}: {resp.text[:200]}"
    places = resp.json().get("places", [])
    out = {"restaurant_name_normalized": row["restaurant_name_normalized"], "district_norm": row["district_norm"],
           "query_text": query}
    if not places:
        return {**out, "match_status": "no_result"}, None
    status = "not_food"
    for p in places:
        types = set(p.get("types", []))
        if not (types & FOOD_TYPES or any(t.endswith("_restaurant") for t in types)):
            continue
        sim = name_similarity(row["restaurant_name"], (p.get("displayName") or {}).get("text"))
        if sim < MIN_NAME_SIMILARITY:
            status = "name_mismatch"
            continue
        loc = p.get("location") or {}
        return {**out, "match_status": "matched", "place_id": p.get("id"),
                "place_name": (p.get("displayName") or {}).get("text"),
                "formatted_address": p.get("formattedAddress"),
                "ward": (component(p, "sublocality_level_1", "administrative_area_level_3", "sublocality")
                         or ward_from_address(p.get("formattedAddress"))),
                "city": component(p, "administrative_area_level_1"),
                "latitude": loc.get("latitude"), "longitude": loc.get("longitude"),
                "place_types": sorted(types), "name_similarity": round(sim, 3)}, None
    return {**out, "match_status": status}, None


results, errors, api_calls = [], [], 0
for row in todo[:budget]:
    result, error = geocode(row.asDict())
    api_calls += 1
    if error and error.startswith(("HTTP 401", "HTTP 403")):
        raise RuntimeError(f"Places API rejected the key (check secret maps/api_key and the API restrictions): {error}")
    if error and error.startswith("HTTP 429"):
        # Quota used up (e.g. 'SearchTextRequest per day'): stop here, the rest is looked up on a later run
        errors.append(f"{row['restaurant_name']}: {error}")
        print(f"⏸️ Places API quota reached after {api_calls} call(s); {len(todo) - len(results)} key(s) wait for the next run")
        break
    if error:
        errors.append(f"{row['restaurant_name']}: {error}")
    else:
        results.append(result)
print(f"API calls: {api_calls} · cached results: {len(results)} · errors (retried next run): {len(errors)}")
for e in errors[:5]:
    print("  ⚠️", e[:300])
if len(todo) > budget:
    print(f"⏸️ Daily cap: {len(todo) - budget} key(s) wait for tomorrow")

# Record the calls first, so the daily cap holds even if a later step fails
spark.sql(f"""
    INSERT INTO {CALLS_TABLE}
    VALUES (current_timestamp(), DATE'{today}', {api_calls}, {len(results)}, {len(errors)})
""")


In [ ]:
# --- MERGE new lookups into the cache (insert-only on the key) ---
from pyspark.sql.functions import current_timestamp

schema = spark.table(GEOCODE_TABLE).schema
if results:
    rows = [{f.name: r.get(f.name) for f in schema if f.name != "geocoded_at"} for r in results]
    new_df = spark.createDataFrame(rows, schema=", ".join(
        f"{f.name} {f.dataType.simpleString()}" for f in schema if f.name != "geocoded_at"))
    new_df.withColumn("geocoded_at", current_timestamp()).createOrReplaceTempView("geocode_src")
    spark.sql(f"""
        MERGE INTO {GEOCODE_TABLE} AS t
        USING geocode_src AS s
        ON t.restaurant_name_normalized = s.restaurant_name_normalized AND t.district_norm <=> s.district_norm
        WHEN NOT MATCHED THEN INSERT *
    """)
print(f"✅ Merged {len(results)} lookups into {GEOCODE_TABLE}")

# Rows cached before the ward was parsed from the address text (no API call needed)
spark.sql(f"""
    UPDATE {GEOCODE_TABLE}
    SET ward = trim(element_at(split(formatted_address, ','), -2))
    WHERE ward IS NULL AND match_status = 'matched' AND size(split(formatted_address, ',')) >= 3
""")


In [ ]:
# --- Verify ---
stats = spark.sql(f"""
    SELECT COUNT(*) AS n,
           COUNT(DISTINCT restaurant_name_normalized, coalesce(district_norm, '')) AS n_keys,
           COUNT_IF(match_status = 'matched') AS matched,
           COUNT(DISTINCT place_id) AS places
    FROM {GEOCODE_TABLE}
""").collect()[0]
print(f"📊 {GEOCODE_TABLE}: {stats['n']} keys ({stats['matched']} matched → {stats['places']} distinct places)")
print(f"📊 Grain check (name × district): {stats['n']} rows vs {stats['n_keys']} distinct "
      + ("✅" if stats["n"] == stats["n_keys"] else "❌ duplicates"))
display(spark.sql(f"SELECT match_status, COUNT(*) AS n FROM {GEOCODE_TABLE} GROUP BY 1 ORDER BY 2 DESC"))
dbutils.jobs.taskValues.set("api_calls", api_calls)
print("\n--- stg_restaurant_geocode complete ---")
